# VAN 전체 API 조회 테스트
VAN 탭의 8개 API를 모두 GET으로 테스트합니다. 발급·취소 요청은 하지 않습니다.
각 셀은 독립적으로 오류를 기록하며, 실패는 0건과 구분합니다. 매출·입금 상세는 전체 페이지를 조회합니다.
입금 API의 날짜는 입금일자, 보류는 승인일자, 청구는 청구일자입니다. 입금 API에는 단말기 필터가 없으므로 사업자 범위 결과입니다.
공개 명세: https://exttran.smilebiz.co.kr/getApiSvcInfoData?SVC_CTGR=VAN


## 0. 공통 설정

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display
# 노트북 폴더 또는 저장소 루트에서 실행할 수 있습니다.
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = [parent, parent / "smilebiz-van/van-api"]
    helper_dir = next((p for p in candidates if (p / "van_test_helpers.py").exists()), None)
    if helper_dir is not None:
        sys.path.insert(0, str(helper_dir))
        break
else:
    raise RuntimeError("노트북 폴더 또는 저장소 루트에서 실행하세요.")
from van_test_helpers import van_get, rows_of, fetch_pages, sales_params, env_value
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

from datetime import datetime, timezone, timedelta
SDATE = datetime.now(timezone(timedelta(hours=9))).strftime("%Y%m%d")
EDATE = SDATE
# 특정 가맹점만 좁혀 조회할 때 .env에 채운다. 비우면 사업자 전체 범위로 조회한다.
COMP_NO = env_value("SMARTRO_VAN_COMP_NO")
COMP_IDX = env_value("SMARTRO_VAN_COMP_IDX")
TERMID = env_value("SMARTRO_VAN_TERMID")
ACQHID = ""
TEST_RESULTS = {}
def run_test(name, path, params=None, paged=False):
    TEST_RESULTS.pop(name, None)
    try:
        if paged:
            rows, raw = fetch_pages(path, params)
        else:
            data = van_get(path, params)
            rows, raw = rows_of(data), [data]
        TEST_RESULTS[name] = {"status": "완료", "rows": rows, "raw": raw}
        print(name, "완료", f"{len(rows)}건" if rows or paged else "")
        if rows:
            display(pd.DataFrame(rows))
        elif not paged:
            display(raw[0])
        else:
            print("조회 내역 없음")
    except RuntimeError as error:
        TEST_RESULTS[name] = {"status": "오류", "error": str(error)}
        print(name, "오류:", error)


## 1. 서버연결 상태체크

In [ ]:
run_test("서버체크", "/V1/common/serverChecks")

## 2. 공통코드정보조회

In [ ]:
run_test("공통코드", "/V1/common/getCommCodeInfo")

## 3. 매출집계 조회

In [ ]:
run_test("매출집계", "/V1/sales/getSalesSum", dict(SDATE=SDATE, EDATE=EDATE, COMP_NO=COMP_NO, COMP_IDX=COMP_IDX, TERMID=TERMID))

## 4. 매출내역 조회 — 결제수단별 전체 페이지

In [ ]:
comm = TEST_RESULTS.get("공통코드", {})
if comm.get("status") != "완료":
    raise RuntimeError("2번 공통코드 조회를 먼저 완료하세요.")
gbn_codes = [str(c.get("CODE", c.get("ITEM_CODE")))
             for g in comm["raw"][0].get("CODE_INFO", []) if g.get("GROUP_CODE") == "GBN"
             for c in g.get("CODES", []) if c.get("DEL_FLAG") != "Y"]
if not gbn_codes:
    raise RuntimeError("조회할 결제수단 코드가 없습니다.")
for gbn in gbn_codes:
    run_test("매출내역 GBN=" + gbn, "/V1/sales/getSalesList",
             sales_params(SDATE, EDATE, TERMID, gbn, comp_no=COMP_NO, comp_idx=COMP_IDX), paged=True)


## 5. 입금내역 조회(집계)

In [ ]:
run_test("5. 입금내역 조회(집계)", "/V1/deposit/getDepositList", dict(SDATE=SDATE, EDATE=EDATE, COMP_NO=COMP_NO, COMP_IDX=COMP_IDX, ACQHID=ACQHID), paged=False)

## 6. 입금내역 조회(상세)

In [ ]:
run_test("6. 입금내역 조회(상세)", "/V1/deposit/getDepositDetail", dict(SDATE=SDATE, EDATE=EDATE, COMP_NO=COMP_NO, COMP_IDX=COMP_IDX, ACQHID=ACQHID), paged=True)

## 7. 입금보류내역 조회

In [ ]:
run_test("7. 입금보류내역 조회", "/V1/deposit/getDepositPndList", dict(SDATE=SDATE, EDATE=EDATE, COMP_NO=COMP_NO, COMP_IDX=COMP_IDX, ACQHID=ACQHID), paged=True)

## 8. 청구내역 조회

In [ ]:
run_test("8. 청구내역 조회", "/V1/deposit/getDepositBillList", dict(SDATE=SDATE, EDATE=EDATE, COMP_NO=COMP_NO, COMP_IDX=COMP_IDX, ACQHID=ACQHID), paged=True)

## 9. 테스트 결과 요약

In [ ]:
display(pd.DataFrame([{ "조회": name, "상태": result["status"], "건수": len(result["rows"]) if "rows" in result else None, "오류": result.get("error", "") } for name, result in TEST_RESULTS.items()]))